<a href="https://colab.research.google.com/github/zorGizem/Erken-Evre-Alzhemir-Tespiti/blob/main/notebooks/ND_Bias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Manyetik Alan Düzeltme(Bias)
MR görüntüleme sürecinde, manyetik alanın homojen olmaması nedeniyle görüntülerde "Bias Field" adı verilen istenmeyen düşük frekanslı sinyaller oluşabilmektedir. Bu durum, görüntünün farklı bölgelerinde yapay parlaklık farklarına yol açarak anatomik dokuların (gri madde, beyaz madde vb.) yanlış analiz edilmesine sebep olabilir. Alzheimer teşhisi gibi doku yoğunluğu ve hacmi üzerine kurulu hassas analizlerde, bu parlaklık düzensizliklerinin giderilmesi kritik bir öneme sahiptir.

# Neden ANTs (N4ITK) Tercih Edildi?
Görüntüdeki yapay parlaklık farklarını gidermek için kullanılan N4ITK algoritması, tıbbi görüntüleme literatüründeki en güvenilir yöntemdir. ANTs kütüphanesi aracılığıyla uyguladığımız bu yöntem; shrink_factor=4 ile yüksek işlem hızı, get_mask fonksiyonu ile sadece kafa içi dokulara odaklanan hassas düzeltme ve sonraki işleme adımlarıyla tam veri uyumu sunduğu için tercih edilmiştir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

CONFIG_CN = {
    "kategori": "CN",
    "kaynak": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/nifti_dataset/nifti_CN',
    "hedef": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/n4_bias/n4_CN'
}


In [ ]:
CONFIG_EMCI = {
    "kategori": "EMCI",
    "kaynak": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/nifti_dataset/nifti_EMCI',
    "hedef": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/n4_bias/n4_EMCI'
}

In [ ]:

!pip install antspyx -q

In [ ]:
import os
import shutil
import ants

In [ ]:

def nifti_bias_pipeline(config):
    source_base = config["kaynak"]
    output_base = config["hedef"]
    kategori = config["kategori"]

    # Geçici çalışma alanları
    temp_input_dir = f'/content/temp_bias_in_{kategori}'
    temp_output_dir = f'/content/temp_bias_out_{kategori}'

    # İstatistikler
    stats = {'yeni_hasta': 0, 'atlanan_hasta': 0, 'yeni_seans': 0, 'atlanan_seans': 0, 'hatali': 0}

    # Klasör hazırlığı
    os.makedirs(output_base, exist_ok=True)
    for t_dir in [temp_input_dir, temp_output_dir]:
        if os.path.exists(t_dir): shutil.rmtree(t_dir)
        os.makedirs(t_dir)

    print(f"\n {kategori}  Bias (N4) İşlemi Başlıyor.")
    print("-" * 80)

    if not os.path.exists(source_base):
        print(f" Hata: Kaynak yol bulunamadı! {source_base}")
        return

    # Hastaları say ve listele
    subjects = sorted([s for s in os.listdir(source_base) if os.path.isdir(os.path.join(source_base, s))])
    toplam_hasta = len(subjects)

    print(f" Klasörde toplam {toplam_hasta} hasta bulundu. Tarama başlıyor.")

    hasta_sirasi = 1

    for subject_id in subjects:
        subject_path = os.path.join(source_base, subject_id)
        print(f"\n Hasta [{hasta_sirasi}/{toplam_hasta}]: {subject_id}")
        hasta_sirasi += 1

        seanslar = sorted(os.listdir(subject_path))
        yeni_islem_yapildi = False
        tum_seanslar_atlandi = True

        for seans_tarihi in seanslar:
            seans_yolu = os.path.join(subject_path, seans_tarihi)
            if not os.path.isdir(seans_yolu): continue

            hedef_seans_yolu = os.path.join(output_base, subject_id, seans_tarihi)

            # Atlanan (zaten yapılmış) kontrolü
            if os.path.exists(hedef_seans_yolu) and any(f.endswith('.nii.gz') for f in os.listdir(hedef_seans_yolu)):
                print(f"   [Atlandı] Seans: {seans_tarihi}")
                stats['atlanan_seans'] += 1
                continue

            nifti_dosyalari = [f for f in os.listdir(seans_yolu) if f.endswith('.nii.gz')]
            if not nifti_dosyalari:
                continue

            ana_nifti_adi = nifti_dosyalari[0]
            orijinal_nifti_yolu = os.path.join(seans_yolu, ana_nifti_adi)

            tum_seanslar_atlandi = False
            os.makedirs(hedef_seans_yolu, exist_ok=True)
            print(f"   [İşleniyor] Seans: {seans_tarihi} ...", end="")

            # Temp klasör temizliği
            for t_dir in [temp_input_dir, temp_output_dir]:
                for f in os.listdir(t_dir): os.remove(os.path.join(t_dir, f))

            local_girdi = os.path.join(temp_input_dir, ana_nifti_adi)
            local_cikti = os.path.join(temp_output_dir, ana_nifti_adi)

            try:
                shutil.copy(orijinal_nifti_yolu, local_girdi)
                img = ants.image_read(local_girdi)


                #  Görüntüdeki siyah boşlukları atıp sadece kafayı bulan bir maske çıkar.Beynin bulunduğu dosyadaki beyni maskeler.böylelikle beyinin dışındaki siyah bölge görülmez ve böylelikle bias işlemi yapılırken sadece beyin üzerind eişlem yapılıp zamnadan kazanç sağlanır.
                maske = ants.get_mask(img)

                # shrink_factor=4: İşlemi hızlandırmak ve RAM kullanımını optimize etmek için kullanılır.
                # Düşük çözünürlükte hesaplanan bias sinyali, detay kaybı olmadan ana görüntüye uygulanır.
                n4_img = ants.n4_bias_field_correction(img, mask=maske, shrink_factor=4)


                # Temizlenen dosyayı Drive'a orijinal ismiyle kaydet
                ants.image_write(n4_img, local_cikti)
                shutil.move(local_cikti, os.path.join(hedef_seans_yolu, ana_nifti_adi))

                print("  Başarılı")
                stats['yeni_seans'] += 1
                yeni_islem_yapildi = True

            except Exception as e:
                print(f"  Hata: {str(e)}")
                stats['hatali'] += 1

        if yeni_islem_yapildi: stats['yeni_hasta'] += 1
        elif tum_seanslar_atlandi and len(seanslar) > 0: stats['atlanan_hasta'] += 1

    print("\n" + "="*80)
    print(f" {kategori} GRUBU BİAS (N4) ÖZETİ")
    print("-" * 80)
    print(f"  - Toplam Bulunan Hasta : {toplam_hasta}")
    print(f"  - Yeni İşlenen Hasta   : {stats['yeni_hasta']}")
    print(f"  - Atlanan (Biten) Hasta: {stats['atlanan_hasta']}")
    print("="*80)

In [ ]:
nifti_bias_pipeline(CONFIG_CN)

In [ ]:
nifti_bias_pipeline(CONFIG_EMCI)